# 3.5 归一化 (Normalization)

归一化稳定训练过程中的数值分布，是深度Transformer训练的关键组件。

本节涵盖：
- RMSNorm
- LayerNorm
- DeepNorm

## 1. RMSNorm

**目的**：更高效的层归一化

**基本原理**：移除LayerNorm中的均值中心化操作，仅计算均方根进行缩放，计算量更少且效果相当。代表：LLaMA、Mistral。

**公式**：RMSNorm(x) = x / RMS(x) * γ，其中RMS(x) = sqrt(mean(x²))

**与LayerNorm的区别**：
- LayerNorm: (x - μ) / σ * γ + β（需要计算均值和方差）
- RMSNorm: x / RMS(x) * γ（仅计算均方根，无均值中心化，无偏置β）

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight

x = torch.randn(4, 10, 64) * 3 + 2
rms_norm = RMSNorm(64)
layer_norm = nn.LayerNorm(64)

x_rms = rms_norm(x)
x_ln = layer_norm(x)

print('=== RMSNorm vs LayerNorm ===')
print(f'Input: mean={x.mean():.3f}, std={x.std():.3f}, rms={torch.sqrt((x**2).mean()):.3f}')
print(f'\nAfter RMSNorm: mean={x_rms.mean():.3f}, std={x_rms.std():.3f}, rms={torch.sqrt((x_rms**2).mean()):.3f}')
print(f'After LayerNorm: mean={x_ln.mean():.3f}, std={x_ln.std():.3f}, rms={torch.sqrt((x_ln**2).mean()):.3f}')

rms_params = sum(p.numel() for p in rms_norm.parameters())
ln_params = sum(p.numel() for p in layer_norm.parameters())
print(f'\nRMSNorm parameters: {rms_params} (weight only)')
print(f'LayerNorm parameters: {ln_params} (weight + bias)')
print(f'\nKey: RMSNorm is simpler and faster than LayerNorm, with comparable performance.')
print(f'It removes the mean-centering step, saving computation while maintaining effectiveness.')

## 2. LayerNorm

**目的**：标准层归一化

**基本原理**：对每个样本的特征维度计算均值和方差进行归一化，稳定训练过程。代表：GPT系列。

**Pre-Norm vs Post-Norm**：
- **Post-Norm**（原始Transformer）：x = LayerNorm(x + Sublayer(x))，训练不稳定但效果上限高
- **Pre-Norm**（GPT-2+）：x = x + Sublayer(LayerNorm(x))，训练更稳定，现代LLM主流选择

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class PreNormTransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, 256), nn.GELU(), nn.Linear(256, d_model))

    def forward(self, x):
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x

class PostNormTransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, 256), nn.GELU(), nn.Linear(256, d_model))

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x

x = torch.randn(2, 10, 64)
pre_norm = PreNormTransformerBlock()
post_norm = PostNormTransformerBlock()

out_pre = pre_norm(x)
out_post = post_norm(x)

print('=== Pre-Norm vs Post-Norm ===')
print(f'Input: mean={x.mean():.3f}, std={x.std():.3f}')
print(f'Pre-Norm output: mean={out_pre.mean():.3f}, std={out_pre.std():.3f}')
print(f'Post-Norm output: mean={out_post.mean():.3f}, std={out_post.std():.3f}')

print(f'\nPre-Norm: normalize BEFORE attention/FFN (x + Sublayer(Norm(x)))')
print(f'Post-Norm: normalize AFTER residual (Norm(x + Sublayer(x)))')
print(f'\nKey: Pre-Norm is the standard in modern LLMs (GPT, LLaMA) for training stability.')

## 3. DeepNorm

**目的**：支持超深Transformer训练

**基本原理**：在Post-Norm基础上对残差连接进行幅度缩放，使超深网络（1000+层）训练更稳定。

**DeepNorm公式**：
- x = DeepNorm(x + α * Sublayer(x))
- DeepNorm(x) = LayerNorm(x) * β
- α = (2N)^(1/4)，β = (2N)^(-1/4)，N为层数

**关键创新**：通过α和β的缩放，确保深层网络中残差分支的信号不会消失或爆炸

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

class DeepNormLayerNorm(nn.Module):
    def __init__(self, dim, N):
        super().__init__()
        self.ln = nn.LayerNorm(dim)
        self.beta = (2 * N) ** (-0.25)

    def forward(self, x):
        return self.ln(x) * self.beta

class DeepNormBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, N=12):
        super().__init__()
        self.alpha = (2 * N) ** 0.25
        self.deep_norm1 = DeepNormLayerNorm(d_model, N)
        self.deep_norm2 = DeepNormLayerNorm(d_model, N)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, 256), nn.GELU(), nn.Linear(256, d_model))

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.deep_norm1(x + self.alpha * attn_out)
        x = self.deep_norm2(x + self.alpha * self.ffn(x))
        return x

print('=== DeepNorm Scaling Factors ===')
for N in [12, 24, 64, 128, 1000]:
    alpha = (2 * N) ** 0.25
    beta = (2 * N) ** (-0.25)
    print(f'N={N:4d} layers: alpha={alpha:.4f}, beta={beta:.4f}, alpha*beta={alpha*beta:.4f}')

x = torch.randn(2, 10, 64)
deep_block = DeepNormBlock(d_model=64, N=1000)
out = deep_block(x)
print(f'\nDeepNorm (N=1000) output: mean={out.mean():.3f}, std={out.std():.3f}')
print(f'\nKey: DeepNorm scales residual branches by alpha and output by beta,')
print(f'ensuring signal stability in very deep (1000+ layer) Transformers.')